## 1. Configuración del archivo de logs
Todos los eventos y errores se guardan en `software_fj.log`. Es el **único** archivo que usa el sistema
(la información de clientes, servicios y reservas vive en objetos y listas).

In [ ]:
# Librerias que se necesitan
import logging
import re
from abc import ABC, abstractmethod
from datetime import datetime
from functools import singledispatchmethod

# Archivo donde se guardan los eventos y los errores
ARCHIVO_LOG = "software_fj.log"

logger = logging.getLogger("SoftwareFJ")
logger.setLevel(logging.DEBUG)  # para que guarde todos los niveles
logger.handlers.clear()  # si corro la celda otra vez no se repiten los mensajes

# el modo "w" borra el log anterior cada vez que se ejecuta
manejador = logging.FileHandler(ARCHIVO_LOG, mode="w", encoding="utf-8")
formato = logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
manejador.setFormatter(formato)
logger.addHandler(manejador)

logger.info("===== Inicio del Sistema Software FJ =====")
print(f"Logger configurado. Los eventos se guardan en '{ARCHIVO_LOG}'.")

Logger configurado. Los eventos se guardan en 'software_fj.log'.


## 2. Excepciones personalizadas
Se crea una jerarquía propia: todas heredan de `SoftwareFJError`, así el sistema puede capturar
cualquier error del dominio con un solo `except` sin ocultar errores inesperados de Python.

In [ ]:
# Excepcion principal, las demas heredan de esta
class SoftwareFJError(Exception):

    def __init__(self, mensaje, codigo="FJ-000"):
        super().__init__(mensaje)
        self.codigo = codigo  # codigo para saber que tipo de error es

    def __str__(self):
        # muestra el codigo y el mensaje juntos
        return f"[{self.codigo}] {self.args[0]}"


# Cuando los datos del cliente estan mal
class ClienteInvalidoError(SoftwareFJError):
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-100")


# Cuando el servicio esta mal creado (tarifa, capacidad, etc)
class ServicioInvalidoError(SoftwareFJError):
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-200")


# Cuando el servicio existe pero no se puede reservar
class ServicioNoDisponibleError(SoftwareFJError):
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-210")


# Cuando la reserva tiene datos incorrectos
class ReservaInvalidaError(SoftwareFJError):
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-300")


# Cuando se intenta algo que no se puede, como cancelar una reserva ya procesada
class OperacionNoPermitidaError(SoftwareFJError):
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-400")


# Cuando un calculo da un resultado que no tiene sentido
class CalculoInconsistenteError(SoftwareFJError):
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-500")


# Muestro los nombres de las excepciones creadas
nombres = []
for clase in SoftwareFJError.__subclasses__():
    nombres.append(clase.__name__)
print("Excepciones personalizadas definidas:", nombres)

Excepciones personalizadas definidas: ['ClienteInvalidoError', 'ServicioInvalidoError', 'ServicioNoDisponibleError', 'ReservaInvalidaError', 'OperacionNoPermitidaError', 'CalculoInconsistenteError']


## 3. Clase abstracta `EntidadBase` y clase `Cliente`
`EntidadBase` representa cualquier entidad general del sistema (cliente, servicio, reserva): le da un
identificador único y una fecha de creación, y **obliga** a las hijas a implementar `describir()` y `validar()`.

`Cliente` aplica **encapsulación**: los atributos son privados (`__nombre`, `__documento`…) y solo se
modifican mediante *properties* que validan cada dato.

In [ ]:
class EntidadBase(ABC):
    # Clase abstracta, de aca heredan cliente, servicio y reserva

    _contador = 0  # se usa para que cada objeto tenga un id diferente

    def __init__(self, prefijo):
        EntidadBase._contador = EntidadBase._contador + 1
        # el id queda por ejemplo CLI-001 o RES-010
        self._id = f"{prefijo}-{EntidadBase._contador:03d}"
        self._fecha_creacion = datetime.now()

    @property
    def id(self):
        # solo se puede leer, no tiene setter
        return self._id

    # Estos dos metodos los tienen que hacer las clases hijas
    @abstractmethod
    def describir(self):
        pass

    @abstractmethod
    def validar(self):
        pass

    def __str__(self):
        # asi el print muestra lo que devuelve describir
        return self.describir()


class Cliente(EntidadBase):
    # Clase del cliente, los datos son privados y se validan en los setter

    PATRON_EMAIL = re.compile(r"^[\w.+-]+@[\w-]+\.[\w.-]+$")

    def __init__(self, nombre, documento, email, telefono):
        super().__init__("CLI")
        # al asignar aca se llama al setter de cada uno y ahi se valida
        self.nombre = nombre
        self.documento = documento
        self.email = email
        self.telefono = telefono

    # nombre
    @property
    def nombre(self):
        return self.__nombre

    @nombre.setter
    def nombre(self, valor):
        if not isinstance(valor, str) or valor.strip() == "":
            raise ClienteInvalidoError("El nombre es obligatorio.")

        # reviso letra por letra que no tenga numeros
        for letra in valor:
            if letra.isdigit():
                raise ClienteInvalidoError(f"El nombre '{valor}' no puede contener números.")

        self.__nombre = valor.strip().title()

    # documento
    @property
    def documento(self):
        return self.__documento

    @documento.setter
    def documento(self, valor):
        valor = str(valor).strip()  # por si llega como numero
        if not valor.isdigit() or len(valor) < 6 or len(valor) > 10:
            raise ClienteInvalidoError(f"Documento '{valor}' inválido (6 a 10 dígitos).")
        self.__documento = valor

    # email
    @property
    def email(self):
        return self.__email

    @email.setter
    def email(self, valor):
        if not isinstance(valor, str) or not self.PATRON_EMAIL.match(valor):
            raise ClienteInvalidoError(f"Correo '{valor}' con formato inválido.")
        self.__email = valor.lower()

    # telefono
    @property
    def telefono(self):
        return self.__telefono

    @telefono.setter
    def telefono(self, valor):
        valor = str(valor).strip()
        # celular de Colombia: 10 numeros y empieza por 3
        if not re.fullmatch(r"3\d{9}", valor):
            raise ClienteInvalidoError(f"Teléfono '{valor}' inválido (10 dígitos iniciando en 3).")
        self.__telefono = valor

    def validar(self):
        # si el cliente se pudo crear es porque ya paso las validaciones
        return True

    def describir(self):
        # solo muestro los ultimos 3 numeros del documento
        cantidad = len(self.documento) - 3
        doc_oculto = "*" * cantidad + self.documento[-3:]
        return f"Cliente {self.id}: {self.nombre} | Doc {doc_oculto} | {self.email}"


# Prueba para ver que funciona
c_prueba = Cliente("ana pérez", "1012345678", "ana@correo.com", "3101234567")
print(c_prueba)

Cliente CLI-001: Ana Pérez | Doc *******678 | ana@correo.com


## 4. Clase abstracta `Servicio` y servicios especializados
Tres servicios heredan de `Servicio`: **ReservaSala**, **AlquilerEquipo** y **Asesoria**. Cada uno
**sobrescribe** `calcular_costo()`, `describir()` y `validar_parametros()` (polimorfismo).

**Sobrecarga de métodos.** Python no permite dos métodos con el mismo nombre, así que se usan las dos
formas propias del lenguaje:
1. `calcular_costo_total(horas, impuesto=..., descuento=...)` — parámetros opcionales (variantes con/sin impuesto y descuento).
2. `cotizar(...)` con `@singledispatchmethod` — se comporta distinto según el **tipo** del argumento (`int`, `float` o `dict`).

In [ ]:
class Servicio(EntidadBase):
    # Clase abstracta para los servicios

    IVA = 0.19

    def __init__(self, nombre, tarifa_hora, disponible=True):
        super().__init__("SRV")

        if not nombre or str(nombre).strip() == "":
            raise ServicioInvalidoError("El servicio necesita un nombre.")
        if not isinstance(tarifa_hora, (int, float)) or tarifa_hora <= 0:
            raise ServicioInvalidoError(f"Tarifa '{tarifa_hora}' inválida: debe ser un número mayor que 0.")

        self._nombre = str(nombre).strip()
        self._tarifa_hora = float(tarifa_hora)
        self._disponible = disponible

    @property
    def nombre(self):
        return self._nombre

    @property
    def disponible(self):
        return self._disponible

    def cambiar_disponibilidad(self, estado):
        # sirve para activar o desactivar el servicio
        self._disponible = estado
        logger.info(f"{self.id} '{self.nombre}' disponible = {estado}")

    # Estos metodos los hace cada servicio a su manera
    @abstractmethod
    def calcular_costo(self, horas):
        pass

    @abstractmethod
    def validar_parametros(self, horas):
        pass

    def validar(self):
        return self._tarifa_hora > 0

    # Sobrecarga 1: con parametros opcionales
    # calcular_costo_total(h)                  -> con IVA
    # calcular_costo_total(h, impuesto=0)      -> sin impuesto
    # calcular_costo_total(h, descuento=0.10)  -> con IVA y descuento del 10 %
    def calcular_costo_total(self, horas, impuesto=None, descuento=0.0):
        if impuesto is None:
            impuesto = self.IVA

        if descuento < 0 or descuento >= 1:
            raise CalculoInconsistenteError(f"Descuento {descuento:.0%} fuera del rango permitido (0 % a 99 %).")
        if impuesto < 0:
            raise CalculoInconsistenteError("El impuesto no puede ser negativo.")

        # aca se usa el calcular_costo de la clase hija (polimorfismo)
        base = self.calcular_costo(horas)
        total = base * (1 - descuento) * (1 + impuesto)

        if total <= 0:
            raise CalculoInconsistenteError(f"Costo total inconsistente: {total}")
        return round(total, 2)

    # Sobrecarga 2: cambia segun el tipo de dato que le llega
    @singledispatchmethod
    def cotizar(self, arg):
        # si llega un tipo que no es int, float o dict cae aca
        raise ServicioInvalidoError(f"No se puede cotizar con un argumento de tipo {type(arg).__name__}.")

    @cotizar.register
    def _(self, horas: int):
        # ejemplo: cotizar(3)
        return self.calcular_costo_total(horas)

    @cotizar.register
    def _(self, horas: float):
        # ejemplo: cotizar(1.5)
        return self.calcular_costo_total(horas)

    @cotizar.register
    def _(self, opciones: dict):
        # ejemplo: cotizar({"horas": 2, "descuento": 0.1})
        try:
            horas = opciones["horas"]
        except KeyError as error:
            raise ServicioInvalidoError("Falta el parámetro 'horas' en la cotización.") from error

        impuesto = opciones.get("impuesto")
        descuento = opciones.get("descuento", 0.0)
        return self.calcular_costo_total(horas, impuesto, descuento)


class ReservaSala(Servicio):
    # Servicio de salas de reuniones

    def __init__(self, nombre, tarifa_hora, capacidad, disponible=True):
        super().__init__(nombre, tarifa_hora, disponible)
        if not isinstance(capacidad, int) or capacidad <= 0:
            raise ServicioInvalidoError(f"Capacidad '{capacidad}' inválida para la sala.")
        self._capacidad = capacidad

    def validar_parametros(self, horas):
        if not isinstance(horas, (int, float)) or horas < 1 or horas > 8:
            raise ReservaInvalidaError(f"Una sala se reserva entre 1 y 8 horas (se pidió {horas}).")

    def calcular_costo(self, horas):
        self.validar_parametros(horas)
        # las salas de mas de 20 personas valen 20 % mas
        if self._capacidad > 20:
            recargo = 1.2
        else:
            recargo = 1.0
        return self._tarifa_hora * horas * recargo

    def describir(self):
        return f"Sala {self.id}: {self.nombre} ({self._capacidad} personas) - ${self._tarifa_hora:,.0f}/h"


class AlquilerEquipo(Servicio):
    # Servicio de alquiler de equipos (portatiles, video beam, etc)

    def __init__(self, nombre, tarifa_hora, unidades, disponible=True):
        super().__init__(nombre, tarifa_hora, disponible)
        if not isinstance(unidades, int) or unidades <= 0:
            raise ServicioInvalidoError(f"Unidades '{unidades}' inválidas para el equipo.")
        self._unidades = unidades

    def validar_parametros(self, horas):
        if not isinstance(horas, (int, float)) or horas < 1 or horas > 72:
            raise ReservaInvalidaError(f"Un equipo se alquila entre 1 y 72 horas (se pidió {horas}).")

    def calcular_costo(self, horas):
        self.validar_parametros(horas)
        seguro = 15000  # el seguro es un valor fijo
        return self._tarifa_hora * horas * self._unidades + seguro

    def describir(self):
        return f"Equipo {self.id}: {self.nombre} x{self._unidades} - ${self._tarifa_hora:,.0f}/h"


class Asesoria(Servicio):
    # Servicio de asesorias, el precio cambia segun el nivel del asesor

    NIVELES = {"junior": 1.0, "senior": 1.5, "experto": 2.0}

    def __init__(self, nombre, tarifa_hora, nivel, disponible=True):
        super().__init__(nombre, tarifa_hora, disponible)
        if nivel not in self.NIVELES:
            raise ServicioInvalidoError(f"Nivel '{nivel}' no existe. Opciones: {list(self.NIVELES)}")
        self._nivel = nivel

    def validar_parametros(self, horas):
        if not isinstance(horas, (int, float)) or horas < 0.5 or horas > 4:
            raise ReservaInvalidaError(f"Una asesoría dura entre 0.5 y 4 horas (se pidió {horas}).")

    def calcular_costo(self, horas):
        self.validar_parametros(horas)
        multiplicador = self.NIVELES[self._nivel]
        return self._tarifa_hora * horas * multiplicador

    def describir(self):
        return f"Asesoría {self.id}: {self.nombre} (nivel {self._nivel}) - ${self._tarifa_hora:,.0f}/h"


# Prueba del polimorfismo: se llama igual pero cada servicio calcula diferente
servicios_prueba = [
    ReservaSala("Sala Prueba", 50000, 10),
    AlquilerEquipo("Portátil Prueba", 8000, 2),
    Asesoria("Python Prueba", 90000, "senior"),
]

for s in servicios_prueba:
    print(f"{s.describir():<60} -> 2 h = ${s.cotizar(2):,.2f}")

Sala SRV-002: Sala Prueba (10 personas) - $50,000/h          -> 2 h = $119,000.00
Equipo SRV-003: Portátil Prueba x2 - $8,000/h                -> 2 h = $55,930.00
Asesoría SRV-004: Python Prueba (nivel senior) - $90,000/h   -> 2 h = $321,300.00


## 5. Clase `Reserva`
Integra **cliente, servicio, duración y estado**. Estados posibles:
`PENDIENTE → CONFIRMADA → PROCESADA`, o `CANCELADA`.
Aquí se usan los bloques **try/except/else**, **try/except/finally** y el **encadenamiento** (`raise ... from ...`).

In [ ]:
class Reserva(EntidadBase):
    # Clase para las reservas que hacen los clientes

    # Estados que puede tener una reserva
    PENDIENTE = "PENDIENTE"
    CONFIRMADA = "CONFIRMADA"
    CANCELADA = "CANCELADA"
    PROCESADA = "PROCESADA"

    def __init__(self, cliente, servicio, horas):
        super().__init__("RES")

        # Primero reviso que los datos esten bien
        if not isinstance(cliente, Cliente):
            raise ReservaInvalidaError("La reserva requiere un cliente válido.")
        if not isinstance(servicio, Servicio):
            raise ReservaInvalidaError("La reserva requiere un servicio válido.")
        if not servicio.disponible:
            raise ServicioNoDisponibleError("El servicio '" + servicio.nombre + "' no está disponible.")

        servicio.validar_parametros(horas)

        self.cliente = cliente
        self.servicio = servicio
        self.horas = horas
        self.__estado = Reserva.PENDIENTE  # privado para que no lo cambien desde afuera
        self.costo = None

    @property
    def estado(self):
        return self.__estado

    def confirmar(self, descuento=0.0):
        # Solo se puede confirmar si esta pendiente
        if self.__estado != Reserva.PENDIENTE:
            raise OperacionNoPermitidaError(f"No se puede confirmar {self.id}: estado {self.__estado}.")

        try:
            costo = self.servicio.calcular_costo_total(self.horas, descuento=descuento)
        except CalculoInconsistenteError as error:
            # uso "from" para que no se pierda el error original
            raise ReservaInvalidaError(f"No se pudo confirmar {self.id} por un error de cálculo.") from error
        else:
            # entra aca si el calculo salio bien
            self.costo = costo
            self.__estado = Reserva.CONFIRMADA
            logger.info(f"{self.id} CONFIRMADA | {self.cliente.nombre} | {self.servicio.nombre} | ${costo:,.2f}")
            return costo

    def cancelar(self, motivo="Sin motivo"):
        # No se puede cancelar si ya se proceso o ya estaba cancelada
        if self.__estado == Reserva.PROCESADA or self.__estado == Reserva.CANCELADA:
            raise OperacionNoPermitidaError(f"No se puede cancelar {self.id}: ya está {self.__estado}.")

        self.__estado = Reserva.CANCELADA
        logger.warning(f"{self.id} CANCELADA | motivo: {motivo}")

    def procesar(self):
        # Cobra la reserva, tiene que estar confirmada antes
        try:
            if self.__estado != Reserva.CONFIRMADA:
                raise OperacionNoPermitidaError(
                    f"Solo se procesan reservas confirmadas ({self.id} está {self.__estado}).")
            self.__estado = Reserva.PROCESADA
            logger.info(f"{self.id} PROCESADA | cobro ${self.costo:,.2f}")
        except OperacionNoPermitidaError:
            logger.error(f"Fallo al procesar {self.id}")
            raise  # la vuelvo a lanzar para que el sistema la maneje
        finally:
            # esto se ejecuta siempre, con error o sin error
            logger.debug(f"Intento de procesamiento de {self.id} finalizado (estado: {self.__estado}).")

    def validar(self):
        estados = [Reserva.PENDIENTE, Reserva.CONFIRMADA, Reserva.CANCELADA, Reserva.PROCESADA]
        return self.__estado in estados

    def describir(self):
        if self.costo is not None:
            costo = f"${self.costo:,.2f}"
        else:
            costo = "sin calcular"

        texto = f"Reserva {self.id}: {self.cliente.nombre} -> {self.servicio.nombre} "
        texto = texto + f"({self.horas} h) [{self.__estado}] {costo}"
        return texto

## 6. Clase `SistemaGestion`
Administra las **listas internas** de clientes, servicios y reservas. Cada operación está protegida:
si algo falla, el error se registra en el log y **el programa sigue funcionando**.

In [ ]:
class SistemaGestion:
    # Clase que maneja todo el sistema, guarda los datos en listas (no hay base de datos)

    def __init__(self):
        self.clientes = []
        self.servicios = []
        self.reservas = []
        # contadores para el resumen del final
        self.exitosas = 0
        self.fallidas = 0

    def _ejecutar(self, descripcion, funcion, *args, **kwargs):
        # Todas las operaciones pasan por aca para que si hay un error
        # el programa no se caiga y quede guardado en el log
        print(f"\n▶ {descripcion}")
        try:
            resultado = funcion(*args, **kwargs)
        except SoftwareFJError as error:
            # errores del sistema (los que creamos nosotros)
            self.fallidas = self.fallidas + 1

            # si el error viene de otro error (raise from) tambien lo muestro
            if error.__cause__:
                causa = f" | causa: {error.__cause__}"
            else:
                causa = ""

            nombre_error = type(error).__name__
            logger.error(f"{descripcion} -> {nombre_error}: {error}{causa}")
            print(f"  ✖ {nombre_error}: {error}{causa}")
            return None
        except Exception as error:
            # cualquier otro error que no se esperaba
            self.fallidas = self.fallidas + 1
            nombre_error = type(error).__name__
            logger.critical(f"{descripcion} -> ERROR INESPERADO {nombre_error}: {error}")
            print(f"  ✖ Error inesperado controlado: {nombre_error}: {error}")
            return None
        else:
            # si no hubo ningun error
            self.exitosas = self.exitosas + 1
            logger.info(f"{descripcion} -> OK")
            print(f"  ✔ {resultado}")
            return resultado
        finally:
            # esto siempre se ejecuta
            logger.debug(f"Operación finalizada: {descripcion}")

    def registrar_cliente(self, nombre, documento, email, telefono):
        def _registrar():
            # reviso que no haya otro cliente con el mismo documento
            for c in self.clientes:
                if c.documento == str(documento):
                    raise ClienteInvalidoError(f"Ya existe un cliente con documento {documento}.")

            cliente = Cliente(nombre, documento, email, telefono)
            self.clientes.append(cliente)
            return cliente

        return self._ejecutar(f"Registrar cliente '{nombre}'", _registrar)

    def agregar_servicio(self, clase_servicio, *args, **kwargs):
        def _agregar():
            servicio = clase_servicio(*args, **kwargs)
            self.servicios.append(servicio)
            return servicio

        descripcion = f"Crear servicio {clase_servicio.__name__} {args[:1]}"
        return self._ejecutar(descripcion, _agregar)

    def crear_reserva(self, cliente, servicio, horas, descuento=0.0):
        def _reservar():
            reserva = Reserva(cliente, servicio, horas)
            reserva.confirmar(descuento)
            self.reservas.append(reserva)
            return reserva

        # si el cliente no es valido (None) se muestra tal cual
        if isinstance(cliente, Cliente):
            nombre = cliente.nombre
        else:
            nombre = cliente

        return self._ejecutar(f"Reservar para {nombre} ({horas} h)", _reservar)

    def cancelar_reserva(self, reserva, motivo):
        def _cancelar():
            if reserva is None:
                raise ReservaInvalidaError("La reserva indicada no existe.")
            reserva.cancelar(motivo)
            return reserva

        id_reserva = getattr(reserva, "id", None)
        return self._ejecutar(f"Cancelar reserva {id_reserva}", _cancelar)

    def procesar_reserva(self, reserva):
        def _procesar():
            if reserva is None:
                raise ReservaInvalidaError("La reserva indicada no existe.")
            reserva.procesar()
            return reserva

        id_reserva = getattr(reserva, "id", None)
        return self._ejecutar(f"Procesar reserva {id_reserva}", _procesar)

    def resumen(self):
        print("\n" + "=" * 70)
        print(f"RESUMEN | Clientes: {len(self.clientes)} | Servicios: {len(self.servicios)} | "
              f"Reservas: {len(self.reservas)}")
        print(f"Operaciones exitosas: {self.exitosas} | Operaciones fallidas controladas: {self.fallidas}")
        print("=" * 70)
        for r in self.reservas:
            print("  •", r)


sistema = SistemaGestion()
print("Sistema listo:", sistema.__class__.__name__)

Sistema listo: SistemaGestion


## 7. Simulación de operaciones
Se ejecutan más de **10 operaciones completas**: registros válidos e inválidos de clientes, creación
correcta e incorrecta de servicios y reservas exitosas y fallidas. El programa **nunca se detiene**.

### 7.1 Registro de clientes

In [ ]:
# 1 y 2: clientes con datos correctos
cli_ana = sistema.registrar_cliente("Ana Pérez", "1012345678", "ana@correo.com", "3101234567")
cli_luis = sistema.registrar_cliente("Luis Gómez", "79888777", "luis.gomez@mail.co", "3157654321")

# 3: el nombre tiene un numero
cli_mal1 = sistema.registrar_cliente("Carl0s", "123456", "carlos@mail.com", "3001112233")

# 4: el documento tiene letras
cli_mal2 = sistema.registrar_cliente("Marta Ruiz", "12AB", "marta@mail.com", "3001112233")

# 5: el correo no tiene arroba
cli_mal3 = sistema.registrar_cliente("Pedro Díaz", "52000111", "pedro-sin-arroba", "3001112233")

# 6: documento repetido (el mismo de Ana)
cli_dup = sistema.registrar_cliente("Ana Duplicada", "1012345678", "otra@correo.com", "3109998877")


▶ Registrar cliente 'Ana Pérez'
  ✔ Cliente CLI-005: Ana Pérez | Doc *******678 | ana@correo.com

▶ Registrar cliente 'Luis Gómez'
  ✔ Cliente CLI-006: Luis Gómez | Doc *****777 | luis.gomez@mail.co

▶ Registrar cliente 'Carl0s'
  ✖ ClienteInvalidoError: [FJ-100] El nombre 'Carl0s' no puede contener números.

▶ Registrar cliente 'Marta Ruiz'
  ✖ ClienteInvalidoError: [FJ-100] Documento '12AB' inválido (6 a 10 dígitos).

▶ Registrar cliente 'Pedro Díaz'
  ✖ ClienteInvalidoError: [FJ-100] Correo 'pedro-sin-arroba' con formato inválido.

▶ Registrar cliente 'Ana Duplicada'
  ✖ ClienteInvalidoError: [FJ-100] Ya existe un cliente con documento 1012345678.


### 7.2 Creación de servicios

In [ ]:
# 7, 8 y 9: servicios bien creados
sala = sistema.agregar_servicio(ReservaSala, "Sala Innovación", 60000, capacidad=25)
portatil = sistema.agregar_servicio(AlquilerEquipo, "Portátil Dell", 8000, unidades=3)
asesoria = sistema.agregar_servicio(Asesoria, "Asesoría en Python", 90000, nivel="senior")

# 10: tarifa negativa
srv_mal1 = sistema.agregar_servicio(ReservaSala, "Sala Fantasma", -5000, capacidad=10)

# 11: ese nivel no existe
srv_mal2 = sistema.agregar_servicio(Asesoria, "Asesoría Redes", 70000, nivel="maestro")

# 12: falta el parametro unidades
srv_mal3 = sistema.agregar_servicio(AlquilerEquipo, "Video Beam", 12000)


▶ Crear servicio ReservaSala ('Sala Innovación',)
  ✔ Sala SRV-010: Sala Innovación (25 personas) - $60,000/h

▶ Crear servicio AlquilerEquipo ('Portátil Dell',)
  ✔ Equipo SRV-011: Portátil Dell x3 - $8,000/h

▶ Crear servicio Asesoria ('Asesoría en Python',)
  ✔ Asesoría SRV-012: Asesoría en Python (nivel senior) - $90,000/h

▶ Crear servicio ReservaSala ('Sala Fantasma',)
  ✖ ServicioInvalidoError: [FJ-200] Tarifa '-5000' inválida: debe ser un número mayor que 0.

▶ Crear servicio Asesoria ('Asesoría Redes',)
  ✖ ServicioInvalidoError: [FJ-200] Nivel 'maestro' no existe. Opciones: ['junior', 'senior', 'experto']

▶ Crear servicio AlquilerEquipo ('Video Beam',)
  ✖ Error inesperado controlado: TypeError: AlquilerEquipo.__init__() missing 1 required positional argument: 'unidades'


### 7.3 Reservas exitosas y fallidas

In [ ]:
# 13, 14 y 15: reservas que salen bien (la 14 tiene descuento del 10 %)
res1 = sistema.crear_reserva(cli_ana, sala, 3)
res2 = sistema.crear_reserva(cli_luis, portatil, 24, descuento=0.10)
res3 = sistema.crear_reserva(cli_ana, asesoria, 2)

# 16: una asesoria no puede durar 10 horas
res4 = sistema.crear_reserva(cli_luis, asesoria, 10)

# 17: ese cliente no se pudo registrar antes, entonces es None
res5 = sistema.crear_reserva(cli_mal1, sala, 2)

# 18: descuento del 150 %, aca se ve el encadenamiento de errores
res6 = sistema.crear_reserva(cli_luis, sala, 2, descuento=1.5)

# 19: desactivo el servicio y pruebo reservar
asesoria.cambiar_disponibilidad(False)
res7 = sistema.crear_reserva(cli_luis, asesoria, 1)
asesoria.cambiar_disponibilidad(True)  # lo vuelvo a activar


▶ Reservar para Ana Pérez (3 h)
  ✔ Reserva RES-015: Ana Pérez -> Sala Innovación (3 h) [CONFIRMADA] $257,040.00

▶ Reservar para Luis Gómez (24 h)
  ✔ Reserva RES-016: Luis Gómez -> Portátil Dell (24 h) [CONFIRMADA] $632,961.00

▶ Reservar para Ana Pérez (2 h)
  ✔ Reserva RES-017: Ana Pérez -> Asesoría en Python (2 h) [CONFIRMADA] $321,300.00

▶ Reservar para Luis Gómez (10 h)
  ✖ ReservaInvalidaError: [FJ-300] Una asesoría dura entre 0.5 y 4 horas (se pidió 10).

▶ Reservar para None (2 h)
  ✖ ReservaInvalidaError: [FJ-300] La reserva requiere un cliente válido.

▶ Reservar para Luis Gómez (2 h)
  ✖ ReservaInvalidaError: [FJ-300] No se pudo confirmar RES-020 por un error de cálculo. | causa: [FJ-500] Descuento 150% fuera del rango permitido (0 % a 99 %).

▶ Reservar para Luis Gómez (1 h)
  ✖ ServicioNoDisponibleError: [FJ-210] El servicio 'Asesoría en Python' no está disponible.


### 7.4 Procesamiento, cancelación y operaciones no permitidas

In [ ]:
# 20: procesar una reserva confirmada
sistema.procesar_reserva(res1)

# 21: no deja cancelar porque ya se proceso
sistema.cancelar_reserva(res1, "El cliente cambió de idea")

# 22: esta si se puede cancelar
sistema.cancelar_reserva(res3, "Asesor incapacitado")

# 23: no se puede procesar una reserva cancelada
sistema.procesar_reserva(res3)

# 24: res4 no se creo, entonces es None
sistema.procesar_reserva(res4)


▶ Procesar reserva RES-015
  ✔ Reserva RES-015: Ana Pérez -> Sala Innovación (3 h) [PROCESADA] $257,040.00

▶ Cancelar reserva RES-015
  ✖ OperacionNoPermitidaError: [FJ-400] No se puede cancelar RES-015: ya está PROCESADA.

▶ Cancelar reserva RES-017
  ✔ Reserva RES-017: Ana Pérez -> Asesoría en Python (2 h) [CANCELADA] $321,300.00

▶ Procesar reserva RES-017
  ✖ OperacionNoPermitidaError: [FJ-400] Solo se procesan reservas confirmadas (RES-017 está CANCELADA).

▶ Procesar reserva None
  ✖ ReservaInvalidaError: [FJ-300] La reserva indicada no existe.


### 7.5 Sobrecarga de métodos (variantes del cálculo de costo)

In [ ]:
print("Cálculos para la Sala Innovación (3 h):")

# Sobrecarga con parametros opcionales
con_iva = sala.calcular_costo_total(3)
sin_impuesto = sala.calcular_costo_total(3, impuesto=0)
con_descuento = sala.calcular_costo_total(3, descuento=0.15)

print("  Con IVA (por defecto):        $", f"{con_iva:,.2f}")
print("  Sin impuesto:                 $", f"{sin_impuesto:,.2f}")
print("  Con IVA y 15 % de descuento:  $", f"{con_descuento:,.2f}")

# Sobrecarga segun el tipo de dato
print("  cotizar(int 3):               $", f"{sala.cotizar(3):,.2f}")
print("  cotizar(float 2.5):           $", f"{sala.cotizar(2.5):,.2f}")
opciones = {"horas": 3, "descuento": 0.15}
print("  cotizar(dict):                $", f"{sala.cotizar(opciones):,.2f}")

# 25: se manda un texto, ese tipo no sirve
sistema._ejecutar("Cotizar con texto", sala.cotizar, "tres horas")

# 26: al diccionario le faltan las horas
sistema._ejecutar("Cotizar sin 'horas'", sala.cotizar, {"descuento": 0.1})

Cálculos para la Sala Innovación (3 h):
  Con IVA (por defecto):        $ 257,040.00
  Sin impuesto:                 $ 216,000.00
  Con IVA y 15 % de descuento:  $ 218,484.00
  cotizar(int 3):               $ 257,040.00
  cotizar(float 2.5):           $ 214,200.00
  cotizar(dict):                $ 218,484.00

▶ Cotizar con texto
  ✖ ServicioInvalidoError: [FJ-200] No se puede cotizar con un argumento de tipo str.

▶ Cotizar sin 'horas'
  ✖ ServicioInvalidoError: [FJ-200] Falta el parámetro 'horas' en la cotización. | causa: 'horas'


### 7.6 Resumen final y contenido del archivo de logs

In [ ]:
sistema.resumen()
logger.info("===== Fin de la simulación =====")

# Leo el archivo del log para mostrar los errores que quedaron guardados
try:
    archivo = open(ARCHIVO_LOG, encoding="utf-8")
    lineas = archivo.readlines()
    archivo.close()
except FileNotFoundError:
    print("No se encontró el archivo de logs.")
else:
    print(f"\nArchivo '{ARCHIVO_LOG}' con {len(lineas)} registros. Errores y advertencias:\n")
    for linea in lineas:
        # solo muestro las lineas que son error o advertencia
        if "ERROR" in linea or "WARNING" in linea or "CRITICAL" in linea:
            print(linea.rstrip())
finally:
    print("\nFin de la ejecución: la aplicación se mantuvo estable en todo momento.")


RESUMEN | Clientes: 2 | Servicios: 3 | Reservas: 3
Operaciones exitosas: 10 | Operaciones fallidas controladas: 16
  • Reserva RES-015: Ana Pérez -> Sala Innovación (3 h) [PROCESADA] $257,040.00
  • Reserva RES-016: Luis Gómez -> Portátil Dell (24 h) [CONFIRMADA] $632,961.00
  • Reserva RES-017: Ana Pérez -> Asesoría en Python (2 h) [CANCELADA] $321,300.00

Archivo 'software_fj.log' con 64 registros. Errores y advertencias:

2026-11-10 15:24:15 | ERROR    | Registrar cliente 'Carl0s' -> ClienteInvalidoError: [FJ-100] El nombre 'Carl0s' no puede contener números.
2026-11-10 15:24:15 | ERROR    | Registrar cliente 'Marta Ruiz' -> ClienteInvalidoError: [FJ-100] Documento '12AB' inválido (6 a 10 dígitos).
2026-11-10 15:24:15 | ERROR    | Registrar cliente 'Pedro Díaz' -> ClienteInvalidoError: [FJ-100] Correo 'pedro-sin-arroba' con formato inválido.
2026-11-10 15:24:15 | ERROR    | Registrar cliente 'Ana Duplicada' -> ClienteInvalidoError: [FJ-100] Ya existe un cliente con documento 101234

## Conclusión técnica
- **Abstracción:** `EntidadBase` y `Servicio` definen contratos que las hijas deben cumplir.
- **Herencia y polimorfismo:** `ReservaSala`, `AlquilerEquipo` y `Asesoria` sobrescriben `calcular_costo`, `describir` y `validar_parametros`.
- **Encapsulación:** datos del cliente y estado de la reserva son privados y se modifican solo con validación.
- **Sobrecarga:** parámetros opcionales y `singledispatchmethod`.
- **Excepciones:** jerarquía propia, `try/except`, `try/except/else`, `try/except/finally`, `raise ... from` y registro de todo en `software_fj.log`.